# Logbook Uploader

Notebook ini membaca `data.csv`, memuat kredensial dari `.env`, lalu mengirim setiap baris logbook ke Ella Unesa.

In [ ]:
from pathlib import Path
import csv
import os
from datetime import datetime
from http.cookiejar import MozillaCookieJar

import requests

ROOT = Path(".")
ENV_PATH = ROOT / ".env"
BASE_URL = "https://ella.unesa.ac.id/log-book/store"

def load_env(path: Path) -> dict[str, str]:
    if not path.exists():
        raise FileNotFoundError(f"File {path} tidak ditemukan")

    values: dict[str, str] = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values

env = load_env(ENV_PATH)
ID_PENDAFTARAN = env["ID_PENDAFTARAN"]
TOKEN = env["TOKEN"]
CSV_PATH = ROOT / env.get("CSV_FILE", "data.csv")
COOKIE_PATH = ROOT / env.get("COOKIE_JAR", "ella_cookie.txt")
LOG_PATH = ROOT / env.get("LOG_FILE", "logbook_result.log")

print("CSV_PATH:", CSV_PATH)
print("COOKIE_PATH:", COOKIE_PATH)
print("LOG_PATH:", LOG_PATH)

In [ ]:
def load_session(cookie_path: Path) -> requests.Session:
    if not cookie_path.exists():
        raise FileNotFoundError(f"File cookie {cookie_path} tidak ditemukan")

    jar = MozillaCookieJar(str(cookie_path))
    jar.load(ignore_discard=True, ignore_expires=True)

    session = requests.Session()
    session.cookies.update(jar)
    return session

def normalize_value(value: str | None) -> str:
    return "" if value is None else value.strip()

def read_rows(csv_path: Path):
    if not csv_path.exists():
        raise FileNotFoundError(f"File CSV {csv_path} tidak ditemukan")

    with csv_path.open(newline="", encoding="utf-8-sig") as csv_file:
        reader = csv.DictReader(csv_file)
        required = ["nama_kegiatan", "uraian_kegiatan", "tgl_kegiatan", "waktu_kegiatan", "file_unggahan"]

        missing = [field for field in required if field not in reader.fieldnames]
        if missing:
            raise ValueError(f"Kolom CSV tidak lengkap: {', '.join(missing)}")

        for row in reader:
            yield {
                "nama_kegiatan": normalize_value(row.get("nama_kegiatan")),
                "uraian_kegiatan": normalize_value(row.get("uraian_kegiatan")),
                "tgl_kegiatan": normalize_value(row.get("tgl_kegiatan")),
                "waktu_kegiatan": normalize_value(row.get("waktu_kegiatan")),
                "file_unggahan": normalize_value(row.get("file_unggahan")),
            }

session = load_session(COOKIE_PATH)
print("Session siap.")

In [ ]:
def upload_row(session: requests.Session, row: dict[str, str]) -> tuple[int, str]:
    file_path = (ROOT / row["file_unggahan"]).resolve()
    if not file_path.exists():
        return 0, f"File tidak ditemukan: {file_path}"

    data = {
        "_token": TOKEN,
        "id_pendaftaran": ID_PENDAFTARAN,
        "nama_kegiatan": row["nama_kegiatan"],
        "uraian_kegiatan": row["uraian_kegiatan"],
        "tgl_kegiatan": row["tgl_kegiatan"],
        "waktu_kegiatan": row["waktu_kegiatan"],
    }

    with file_path.open("rb") as upload_file:
        files = {"file_unggahan": (file_path.name, upload_file)}
        response = session.post(BASE_URL, data=data, files=files, timeout=60)

    return response.status_code, response.text

def run_uploads():
    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    results = []

    for index, row in enumerate(read_rows(CSV_PATH), start=1):
        print("=" * 60)
        print(f"Mengirim logbook #{index}: {row['nama_kegiatan']} pada {row['tgl_kegiatan']}")

        try:
            status_code, response_text = upload_row(session, row)
            ok = 200 <= status_code < 300
            status_label = f"HTTP {status_code}"
            print(status_label)
            print("Berhasil diproses." if ok else "Request tidak sukses.")
        except requests.RequestException as exc:
            status_code = 0
            response_text = str(exc)
            ok = False
            print("Request gagal:", exc)

        log_line = f"[{datetime.now():%Y-%m-%d %H:%M:%S}] | {row['nama_kegiatan']} | {row['tgl_kegiatan']} | HTTP {status_code}"
        with LOG_PATH.open("a", encoding="utf-8") as log_file:
            log_file.write(log_line + "\n")

        results.append({
            "row": row,
            "status_code": status_code,
            "ok": ok,
            "response_text": response_text,
        })

    print("=" * 60)
    print("Iterasi selesai!")
    return results

results = run_uploads()
len(results)

## Cara pakai

1. Buka notebook ini di VS Code atau Jupyter.
2. Jalankan semua cell dari atas ke bawah.
3. Pastikan `data.csv`, file unggahan di folder `file/`, dan cookie `ella_cookie.txt` sudah sesuai.